# 01 - Exploration of the input data: ERA5, EC-Earth3 historical, hosing runs (and RCM when available)

**Purpose**: know every dataset before any correction:
- integrity (period, gaps, calendar, units, NaN, physically impossible values);
- **raw bias of EC-Earth3 historical against ERA5** (what the downscaling / bias adjustment will have to correct);
- the **climate-change and hosing signals** (what the corrections must **preserve**);
- extremes, dry days, distributions, variability.

Datasets: ERA5 (0.25 deg), EC-Earth3 historical (CMIP6, r1i1p1f1), the four stabilization runs (2025 / 2050, original / hosing),
and the RCM runs as soon as their paths are filled in (notebook 00). Variables: `pr`, `tasmax`, `tasmin`, `dtr = tasmax - tasmin`
and `rsds` (surface solar radiation) **wherever it exists**: every section skips what is missing.

**Periods**: EC-Earth3 historical and ERA5 are compared over the same years, `COMPARE_PERIOD`
(1981-2014; 2005-2014 for `rsds`, ERA5 radiation starting in 2005). The hosing runs are explored over their whole length.

**Domain**: maps, means, profiles and distributions over the **LISFLOOD domain** (masks of notebook 00c, run it first).
For the comparisons, ERA5 statistics are regridded (conservatively) onto the EC-Earth3 grid, so both cover **the same pixels**.

**How it works**: each dataset is read **once** and reduced to compact statistics kept in a cache (`CACHE`, on EOS:
several GB with ERA5). The first run of section 1 is long (ERA5 0.25 deg global: about 1-2 h; each GCM run 10-20 min);
afterwards all the figures are fast. Figures and tables: `001_exploration/outputs/01_exploration/` (not in git).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent, Path.home() / "downscaling"):   # ba_utils.py: notebook folder, parent folder, or the repository
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import dask
from dask.diagnostics import ProgressBar
from scipy import stats
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None

VARS_CORE = cfg["variables"]                                  # pr, tasmax, tasmin (whole pipeline)
VARS_ALL = VARS_CORE + cfg.get("extra_variables", [])         # + rsds where it exists
VARS_PLOT = [v for v in ("pr", "tasmax", "tasmin", "dtr", "rsds") if v in VARS_ALL or v == "dtr"]

LABEL = {"era5": "ERA5", "gcm_historical": "historical",
         "gcm_2025_nohosing": "2025 original", "gcm_2025_hosing": "2025 hosing",
         "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing",
         "rcm_2025_nohosing": "RCM 2025 original", "rcm_2025_hosing": "RCM 2025 hosing",
         "rcm_2050_nohosing": "RCM 2050 original", "rcm_2050_hosing": "RCM 2050 hosing"}
COLORS = {"era5": "tab:green", "gcm_historical": "k", "gcm_2025_nohosing": "tab:blue", "gcm_2025_hosing": "tab:red",
          "gcm_2050_nohosing": "tab:cyan", "gcm_2050_hosing": "tab:orange", "rcm_2025_nohosing": "tab:purple",
          "rcm_2025_hosing": "tab:pink", "rcm_2050_nohosing": "tab:olive", "rcm_2050_hosing": "tab:brown"}
UNITS = {"pr": "mm/day", "tasmax": "degC", "tasmin": "degC", "dtr": "degC", "rsds": "W m-2"}
CMAP = {"pr": "YlGnBu", "tasmax": "RdYlBu_r", "tasmin": "RdYlBu_r", "dtr": "magma_r", "rsds": "YlOrRd"}
DCMAP = {"pr": "BrBG", "tasmax": "RdBu_r", "tasmin": "RdBu_r", "dtr": "PuOr_r", "rsds": "PuOr_r"}

# ERA5 and EC-Earth3 historical are compared over the same years (ERA5 radiation starts in 2005)
COMPARE_PERIOD = {"pr": ("1981", "2014"), "tasmax": ("1981", "2014"), "tasmin": ("1981", "2014"), "rsds": ("2005", "2014")}
N_SAMPLE = 3000          # area-weighted random pixels of the domain for the pooled distributions
QUANTILES = [0.001, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
OVERWRITE_CACHE = False
USE_DOMAIN_MASK = True   # True = LISFLOOD domain (notebook 00c must have been run), False = whole globe
BORDERS_RES = "01M"      # coastlines on the maps (GISCO): '01M' finest, '03M' / '10M' lighter
OUT = bu.explo_dir("01_exploration")                            # figures + tables (next to this notebook, not in git)
CACHE = bu.out_dir(cfg, "exploration_cache")                     # statistics cache on EOS (several GB with ERA5)
print("figures:", OUT, "| cache:", CACHE)

try:                                   # cartopy projection if installed (plain lon/lat axes otherwise)
    import cartopy.crs as ccrs
    PROJ = ccrs.PlateCarree()
except ImportError:
    PROJ = None
BORDERS = bu.borders(cfg, BORDERS_RES)
print("coastlines on the maps:", f"GISCO {BORDERS_RES}, {len(BORDERS)} lines" if BORDERS else "none")

## 0. Datasets, grids and domain masks

In [ ]:
AVAIL, GRIDS = {}, {}
for name in LABEL:
    vs = [v for v in VARS_ALL if bu.dataset_available(cfg, name, v)]
    if not vs:
        print(f"[missing] {LABEL[name]}: no variable available (paths in notebook 00)"); continue
    AVAIL[name] = vs
    GRIDS[name] = bu.read_raw(cfg, name, vs[0]).isel(time=0)
    print(f"{LABEL[name]:18s} {', '.join(vs):26s} grid {GRIDS[name].sizes['lat']} x {GRIDS[name].sizes['lon']}")

if USE_DOMAIN_MASK:
    for g in ("era5", "ecearth3"):
        assert bu.domain_mask_path(cfg, g).exists(), "Domain masks not found: run notebook 00c_land_mask first."
    KNOWN_MASKS = [bu.load_domain_mask(cfg, "era5"), bu.load_domain_mask(cfg, "ecearth3")]

def mask_for(da):
    """Domain mask on the grid of `da`: ERA5 or EC-Earth3 mask of 00c, otherwise (e.g. RCM) the ERA5 mask transferred."""
    if not USE_DOMAIN_MASK:
        return xr.ones_like(da.isel({d: 0 for d in da.dims if d not in ("lat", "lon")}, drop=True), dtype=bool)
    for m in KNOWN_MASKS:
        if bu.same_grid(m, da):
            return m.assign_coords(lat=da.lat, lon=da.lon)
    m = bu.load_domain_mask(cfg, "era5", like=da)
    KNOWN_MASKS.append(m)
    return m

MASKS = {n: mask_for(g) for n, g in GRIDS.items()}
GCM_GRID = GRIDS.get("gcm_historical")
DOMAIN = "LISFLOOD domain" if USE_DOMAIN_MASK else "globe"
for n, m in MASKS.items():
    print(f"{LABEL[n]:18s} {DOMAIN}: {int(m.sum()):,} of {m.size:,} pixels")

## 1. Statistics of each dataset (computed once, then read from the cache)

In a single pass over each dataset (ERA5 and historical: over `COMPARE_PERIOD` of each variable):
`<v>_clim_month` (monthly climatology), `<v>_year_mean` (yearly means), `<v>_year_ext` (yearly max of pr = Rx1day,
of tasmax = TXx, yearly min of tasmin = TNn), `pr_year_dry` / `pr_month_dry` (dry-day fraction, pr < 0.1 mm/day),
`pr_year_sdii` (wet-day intensity), `<v>_q` (pooled quantiles on random pixels of the domain), integrity scalars.

In [ ]:
def sample_points(grid, mask, n, seed=0):
    """Random pixels of the domain, probability proportional to their area (cos lat), same pixels for every run of a grid."""
    rng = np.random.default_rng(seed)
    w = (np.cos(np.deg2rad(grid.lat.values))[:, None] * np.ones((1, grid.sizes["lon"])) * mask.values).ravel()
    idx = rng.choice(w.size, size=min(n, int((w > 0).sum())), replace=False, p=w / w.sum())
    return np.unravel_index(idx, (grid.sizes["lat"], grid.sizes["lon"]))


def run_stats(name):
    path = CACHE / f"{name}_stats_{'domain' if USE_DOMAIN_MASK else 'globe'}.nc"
    if path.exists() and not OVERWRITE_CACHE:
        return xr.open_dataset(path).load()
    print(f"-> computing {LABEL[name]} (reads the whole dataset once)")
    d, attrs = {}, {"dataset": name}
    for v in AVAIL[name]:
        da = bu.read_raw(cfg, name, v)
        if name in ("era5", "gcm_historical"):
            da = bu.subset_time(da, *COMPARE_PERIOD[v])
        d[v] = da
        t = da.indexes["time"]; yrs = np.unique(da.time.dt.year.values)
        attrs.update({f"period_{v}": f"{yrs[0]}-{yrs[-1]}", f"ndays_{v}": int(da.sizes["time"]),
                      f"expected_days_{v}": int(len(yrs) * 365), f"duplicates_{v}": int(pd.Index(t).duplicated().sum()),
                      f"orig_units_{v}": da.attrs["orig_units"], f"orig_calendar_{v}": da.attrs["orig_calendar"],
                      f"orig_lat_desc_{v}": int(da.attrs["orig_lat_desc"]), f"orig_lon_0360_{v}": int(da.attrs["orig_lon_0360"])})
    if "tasmax" in d and "tasmin" in d:
        d["dtr"] = d["tasmax"] - d["tasmin"]
    iy, ix = sample_points(GRIDS[name], MASKS[name], N_SAMPLE)
    pick = dict(lat=xr.DataArray(iy, dims="point"), lon=xr.DataArray(ix, dims="point"))
    out, pts, scal = {}, {}, {}
    for v, da in d.items():
        out[f"{v}_clim_month"] = da.groupby("time.month").mean()
        out[f"{v}_year_mean"] = da.groupby("time.year").mean()
        pts[v] = da.isel(**pick)
        scal[f"nan_frac_{v}"] = da.isnull().mean()
        scal[f"min_{v}"] = da.min(); scal[f"max_{v}"] = da.max()
    for v, how in (("pr", "max"), ("tasmax", "max"), ("tasmin", "min")):
        if v in d:
            out[f"{v}_year_ext"] = getattr(d[v].groupby("time.year"), how)()
    if "pr" in d:
        dry = (d["pr"] < bu.THRESH_PR).where(d["pr"].notnull())
        out["pr_year_dry"] = dry.groupby("time.year").mean()
        out["pr_month_dry"] = dry.groupby("time.month").mean()
        out["pr_year_sdii"] = d["pr"].where(d["pr"] >= bu.THRESH_PR).groupby("time.year").mean()
        scal["frac_pr_negative"] = (d["pr"] < 0).mean()
        scal["frac_pr_below_m0.01"] = (d["pr"] < -0.01).mean()
    if "dtr" in d:
        scal["frac_tasmin_gt_tasmax"] = (d["dtr"] < 0).mean()
    with ProgressBar():
        out, pts, scal = dask.compute(out, pts, scal)
    ds = xr.Dataset({k: v.astype("float32") for k, v in out.items()}).assign_coords(q=QUANTILES)
    for v, p in pts.items():
        a = p.values.ravel(); a = a[np.isfinite(a)]
        ds[f"{v}_q"] = ("q", np.quantile(a, QUANTILES).astype("float32"))
        if v == "pr":
            ds["pr_q_wet"] = ("q", np.quantile(a[a >= bu.THRESH_PR], QUANTILES).astype("float32"))
            attrs["pooled_dry_frac"] = float((a < bu.THRESH_PR).mean())
    ds.attrs = {**attrs, **{k: float(val) for k, val in scal.items()}}
    ds.to_netcdf(path)
    return ds


S = {name: run_stats(name) for name in AVAIL}

# ERA5 statistics regridded (conservative) onto the EC-Earth3 grid: comparisons on the same pixels
E = None
if "era5" in S and GCM_GRID is not None:
    path = CACHE / f"era5_on_gcm_grid_stats_{'domain' if USE_DOMAIN_MASK else 'globe'}.nc"
    if path.exists() and not OVERWRITE_CACHE:
        E = xr.open_dataset(path).load()
    else:
        print("-> regridding the ERA5 statistics onto the EC-Earth3 grid")
        E = xr.Dataset({k: bu.regrid_conservative(v, GCM_GRID.lat.values, GCM_GRID.lon.values).compute()
                        for k, v in S["era5"].data_vars.items() if {"lat", "lon"} <= set(v.dims)})
        E = E.assign_coords(q=S["era5"].q)
        for k, v in S["era5"].data_vars.items():
            if "q" in v.dims:
                E[k] = v
        E.attrs = S["era5"].attrs
        E.to_netcdf(path)
print("datasets:", [LABEL[n] for n in S], "| ERA5 on the EC-Earth3 grid:", E is not None)

In [ ]:
# ---- helpers used by all the figures
def has(ds, v, kind="year_mean"):
    return ds is not None and f"{v}_{kind}" in ds


def seas(ds, v, season, years=None):
    """'ANN' = mean of the yearly means (optionally over `years`), 'DJF' / 'JJA' / ... from the monthly climatology."""
    if season == "ANN":
        y = ds[f"{v}_year_mean"]
        y = y.sel(year=slice(*years)) if years else y
        return y.mean("year")
    months = {"DJF": [12, 1, 2], "JJA": [6, 7, 8], "MAM": [3, 4, 5], "SON": [9, 10, 11]}[season]
    return ds[f"{v}_clim_month"].sel(month=months).mean("month")


def welch_sig(a, b, t_crit=2.0):
    """|Welch t| > t_crit on yearly values (dim 'year', NaN years ignored). Serial correlation ignored: indicative."""
    a, b = a.dropna("year", how="all"), b.dropna("year", how="all")
    se = np.sqrt(a.var("year", ddof=1) / a.sizes["year"] + b.var("year", ddof=1) / b.sizes["year"])
    return np.abs(a.mean("year") - b.mean("year")) / se > t_crit


def diff(a, b, v):
    """a - b; relative (%) for pr, masked where the reference is below 0.1 mm/day."""
    return (100 * (a - b) / b).where(b > 0.1) if v == "pr" else a - b


def dlabel(v):
    return "%" if v == "pr" else UNITS[v]


def dmean(da):
    """Area-weighted mean over the domain (mask of the grid of `da`)."""
    return bu.area_mean(da.where(mask_for(da)))


def map_axes(nr, nc, w=5.0, h=2.7):
    kw = {"subplot_kw": {"projection": PROJ}} if PROJ is not None else {}
    fig, axes = plt.subplots(nr, nc, figsize=(w * nc, h * nr), squeeze=False, **kw)
    return fig, axes


def draw(ax, da, title, cmap="viridis", lim=None, sym=False, sig=None, label=""):
    m = mask_for(da)
    da = da.where(m)                          # outside the domain: grey
    if sig is not None:
        sig = sig & m
    ax.set_facecolor("0.85")
    if sym:
        lim = lim or float(np.nanpercentile(np.abs(da.values), 98)) or 1.0
        vmin, vmax = -lim, lim
    else:
        vmin, vmax = (None, None) if lim is None else lim
    kw = {"transform": PROJ} if PROJ is not None else {}
    im = ax.pcolormesh(da.lon, da.lat, da, shading="auto", cmap=cmap, vmin=vmin, vmax=vmax, **kw)
    if sig is not None:
        ax.contourf(sig.lon, sig.lat, sig.astype(float), levels=[0.5, 1.5], hatches=["...."], colors="none", **kw)
    if BORDERS:
        bu.draw_borders(ax, BORDERS, transform=PROJ)
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8, label=label)


def clim_lim(da, v):
    return (0, float(np.nanpercentile(da.where(mask_for(da)), 98))) if v in ("pr", "rsds") else None


H = S.get("gcm_historical")
VARS_HIST_ERA5 = [v for v in VARS_PLOT if has(H, v) and has(E, v)]
print("variables compared historical vs ERA5:", VARS_HIST_ERA5)

## 2. Inventory and integrity of each dataset

In [ ]:
rows, alerts = [], []
for name, ds in S.items():
    for v in [x for x in VARS_ALL if f"period_{x}" in ds.attrs]:
        a = ds.attrs
        r = {"dataset": LABEL[name], "var": v, "period": a[f"period_{v}"], "n days": a[f"ndays_{v}"],
             "expected (noleap)": a[f"expected_days_{v}"], "duplicated dates": a[f"duplicates_{v}"],
             "orig units": a[f"orig_units_{v}"], "orig calendar": a[f"orig_calendar_{v}"],
             "NaN frac": a[f"nan_frac_{v}"], "min": round(a[f"min_{v}"], 3), "max": round(a[f"max_{v}"], 3),
             f"{DOMAIN} mean": round(float(dmean(seas(ds, v, "ANN"))), 3), "globe mean": round(float(bu.area_mean(seas(ds, v, "ANN"))), 3)}
        rows.append(r)
        k = f"{LABEL[name]}/{v}"
        if r["n days"] != r["expected (noleap)"]:
            alerts.append(f"{k}: {r['n days']} days for {r['expected (noleap)']} expected (gaps or partial years)")
        if r["duplicated dates"]:
            alerts.append(f"{k}: {r['duplicated dates']} duplicated dates")
        if r["NaN frac"] > 0:
            alerts.append(f"{k}: {100 * r['NaN frac']:.3f} % NaN")
        if v in ("tasmax", "tasmin") and not (-95 < r["min"] and r["max"] < 70):
            alerts.append(f"{k}: temperature outside the plausible range ({r['min']} .. {r['max']} degC)")
        if v == "rsds" and (r["min"] < -1 or r["max"] > 500):
            alerts.append(f"{k}: radiation outside the plausible range ({r['min']} .. {r['max']} W m-2)")
        if r["orig calendar"] not in ("noleap", "365_day"):
            alerts.append(f"{k}: calendar '{r['orig calendar']}' converted to noleap (29 February removed)")
    if ds.attrs.get("frac_pr_below_m0.01", 0) > 0:
        alerts.append(f"{LABEL[name]}/pr: {100 * ds.attrs['frac_pr_below_m0.01']:.4f} % of values below -0.01 mm/day")
    if ds.attrs.get("frac_tasmin_gt_tasmax", 0) > 0:
        alerts.append(f"{LABEL[name]}: {100 * ds.attrs['frac_tasmin_gt_tasmax']:.4f} % of days with tasmin > tasmax")
integrity = pd.DataFrame(rows).set_index(["dataset", "var"])
integrity.to_csv(OUT / "integrity.csv")
with pd.option_context("display.width", 250, "display.max_columns", 30):
    display(integrity)
print("\n".join(alerts) if alerts else "No alert.")

Notes: EC-Earth3 has many *tiny* negative precipitation values (about -1e-19 mm/day, numerical zeros), harmless
(below the 0.1 mm/day dry-day threshold): only values below -0.01 mm/day are reported. Calendars and units are converted
automatically when reading (mm/day, degC, W m-2, `noleap`).

## 3. Stationarity: yearly means over the domain and trends

The stabilization runs are meant to be in a (quasi-)stable climate: a trend inside a run matters, as the corrections
are calibrated over the whole 2025 original run and validated on its two halves. ERA5 and the historical run warm.

In [ ]:
vs = [v for v in VARS_PLOT if any(has(ds, v) for ds in S.values())]
fig, axes = plt.subplots(1, len(vs), figsize=(5 * len(vs), 3.8), squeeze=False)
trend_rows = []
for ax, v in zip(axes[0], vs):
    for name, ds in S.items():
        if not has(ds, v):
            continue
        g = dmean(ds[f"{v}_year_mean"]).dropna("year")
        ax.plot(g.year, g, color=COLORS[name], lw=1, label=LABEL[name])
        if g.sizes["year"] > 2:
            lr = stats.linregress(g.year.values, g.values); half = g.sizes["year"] // 2
            trend_rows.append({"dataset": LABEL[name], "var": v, "trend per decade": round(10 * lr.slope, 4),
                               "p-value": round(lr.pvalue, 4), "2nd half - 1st half": round(float(g[half:].mean() - g[:half].mean()), 4)})
    ax.set_title(f"{DOMAIN} mean {v} ({UNITS[v]})"); ax.set_xlabel("year")
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "yearly_means.png", dpi=110); plt.show()
if trend_rows:
    trends = pd.DataFrame(trend_rows).set_index(["dataset", "var"])
    trends.to_csv(OUT / "trends.csv"); display(trends)

## 4. Climatologies: ERA5 (0.25 deg) and EC-Earth3 historical (native grid), same period

In [ ]:
for name in ("era5", "gcm_historical"):
    ds = S.get(name)
    vs = [v for v in VARS_PLOT if has(ds, v)]
    if not vs:
        continue
    fig, axes = map_axes(len(vs), 3)
    for r, v in enumerate(vs):
        for c, s in enumerate(("ANN", "DJF", "JJA")):
            x = seas(ds, v, s)
            draw(axes[r, c], x, f"{LABEL[name]} {v} {s}", cmap=CMAP[v], lim=clim_lim(x, v), label=UNITS[v])
    plt.tight_layout(); plt.savefig(OUT / f"climatology_{name}.png", dpi=110); plt.show()

## 5. Raw bias of EC-Earth3 historical against ERA5 (on the EC-Earth3 grid, same period)

What the downscaling / bias adjustment has to correct. pr in %. Dots: |Welch t| > 2 on the yearly values (annual column).
ERA5 regridded conservatively to 0.7 deg: the resolution difference itself is part of what is seen here.

In [ ]:
if VARS_HIST_ERA5:
    fig, axes = map_axes(len(VARS_HIST_ERA5), 3)
    for r, v in enumerate(VARS_HIST_ERA5):
        sig = welch_sig(H[f"{v}_year_mean"], E[f"{v}_year_mean"])
        for c, s in enumerate(("ANN", "DJF", "JJA")):
            draw(axes[r, c], diff(seas(H, v, s), seas(E, v, s), v), f"historical - ERA5: {v} {s}", cmap=DCMAP[v],
                 sym=True, sig=sig if s == "ANN" else None, label=dlabel(v))
    plt.tight_layout(); plt.savefig(OUT / "bias_historical_minus_era5.png", dpi=110); plt.show()
    rows = [{"var": v, f"{DOMAIN} mean historical": round(float(dmean(seas(H, v, "ANN"))), 3),
             f"{DOMAIN} mean ERA5": round(float(dmean(seas(E, v, "ANN"))), 3),
             "bias": round(float(dmean(seas(H, v, "ANN")) - dmean(seas(E, v, "ANN"))), 3)} for v in VARS_HIST_ERA5]
    display(pd.DataFrame(rows).set_index("var"))
else:
    print("ERA5 or EC-Earth3 historical missing: no bias maps")

## 6. Signal maps (what the corrections must preserve)

Differences of climatologies (pr in %); dots: |Welch t| > 2 on the yearly values (annual column; indicative).
Compare later with the same maps after the downscaling / bias adjustment.

In [ ]:
PAIRS = [("gcm_2025_nohosing", "gcm_historical", "2025 original - historical (climate change)"),
         ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original"),
         ("gcm_2050_nohosing", "gcm_2025_nohosing", "2050 original - 2025 original"),
         ("rcm_2025_hosing", "rcm_2025_nohosing", "RCM 2025 hosing - original"),
         ("rcm_2050_hosing", "rcm_2050_nohosing", "RCM 2050 hosing - original")]
for a, b, lab in PAIRS:
    vs = [v for v in VARS_PLOT if has(S.get(a), v) and has(S.get(b), v)]
    if not vs:
        print(f"[missing] {lab}"); continue
    fig, axes = map_axes(len(vs), 3)
    for r, v in enumerate(vs):
        sig = welch_sig(S[a][f"{v}_year_mean"], S[b][f"{v}_year_mean"])
        for c, s in enumerate(("ANN", "DJF", "JJA")):
            draw(axes[r, c], diff(seas(S[a], v, s), seas(S[b], v, s), v), f"{v} {s}", cmap=DCMAP[v], sym=True,
                 sig=sig if s == "ANN" else None, label=dlabel(v))
    fig.suptitle(lab, y=1.0)
    plt.tight_layout(); plt.savefig(OUT / f"signal_{a}_minus_{b}.png", dpi=110, bbox_inches="tight"); plt.show()

## 7. Zonal means over the domain (annual climatology)

Top: every dataset (ERA5 on the EC-Earth3 grid, same pixels as the GCM). Bottom: historical - ERA5 and hosing - original.

In [ ]:
ONE_D = {**{n: ds for n, ds in S.items() if n != "era5"}, **({"era5": E} if E is not None else {})}
vs = [v for v in VARS_PLOT if any(has(ds, v) for ds in ONE_D.values())]
DIFFS = [("gcm_historical", "era5", "historical - ERA5", "k"), ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original", "tab:red"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original", "tab:orange")]
fig, axes = plt.subplots(2, len(vs), figsize=(5 * len(vs), 7), squeeze=False)
for c, v in enumerate(vs):
    for name, ds in ONE_D.items():
        if has(ds, v):
            z = seas(ds, v, "ANN"); z = z.where(mask_for(z)).mean("lon")
            axes[0, c].plot(z.lat, z, color=COLORS[name], lw=1, label=LABEL[name])
    axes[0, c].set_title(f"zonal mean {v} ({UNITS[v]})")
    for a, b, lab, col in DIFFS:
        if has(ONE_D.get(a), v) and has(ONE_D.get(b), v):
            z = seas(ONE_D[a], v, "ANN") - seas(ONE_D[b], v, "ANN")
            z = z.where(mask_for(z)).mean("lon")
            axes[1, c].plot(z.lat, z, color=col, label=lab)
    axes[1, c].axhline(0, color="grey", lw=0.5); axes[1, c].set_xlabel("latitude"); axes[1, c].set_title(f"differences, zonal {v}")
axes[0, 0].legend(fontsize=7); axes[1, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "zonal_means.png", dpi=110); plt.show()

## 8. Annual cycle (means over the domain; ERA5 on the EC-Earth3 grid)

In [ ]:
fig, axes = plt.subplots(2, len(vs), figsize=(5 * len(vs), 7), squeeze=False)
for c, v in enumerate(vs):
    for name, ds in ONE_D.items():
        if has(ds, v, "clim_month"):
            m = dmean(ds[f"{v}_clim_month"])
            axes[0, c].plot(m.month, m, "o-", ms=3, color=COLORS[name], label=LABEL[name])
    axes[0, c].set_title(f"annual cycle, {DOMAIN} {v} ({UNITS[v]})")
    for a, b, lab, col in DIFFS:
        if has(ONE_D.get(a), v, "clim_month") and has(ONE_D.get(b), v, "clim_month"):
            m = dmean(ONE_D[a][f"{v}_clim_month"]) - dmean(ONE_D[b][f"{v}_clim_month"])
            axes[1, c].plot(m.month, m, "s-", ms=3, color=col, label=lab)
    axes[1, c].axhline(0, color="grey", lw=0.5); axes[1, c].set_xlabel("month")
axes[0, 0].legend(fontsize=7); axes[1, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "annual_cycle.png", dpi=110); plt.show()

## 9. Extremes: Rx1day (yearly max pr), TXx (yearly max tasmax), TNn (yearly min tasmin)

Rows: historical climatology; historical - ERA5 (raw bias; ERA5 maxima regridded, so partly a resolution effect);
hosing - original. Rx1day in %; dots = |Welch t| > 2.

In [ ]:
EXT = {"pr": "Rx1day", "tasmax": "TXx", "tasmin": "TNn"}
ROWS = [("gcm_historical", None, "historical"), ("gcm_historical", "era5", "historical - ERA5"),
        ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"), ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original")]
ROWS = [r for r in ROWS if r[0] in ONE_D and (r[1] is None or r[1] in ONE_D)]
evs = [v for v in EXT if any(has(ONE_D[r[0]], v, "year_ext") for r in ROWS)]
if ROWS and evs:
    fig, axes = map_axes(len(ROWS), len(evs))
    for r, (a, b, lab) in enumerate(ROWS):
        for c, v in enumerate(evs):
            if not has(ONE_D[a], v, "year_ext") or (b and not has(ONE_D[b], v, "year_ext")):
                axes[r, c].set_visible(False); continue
            x = ONE_D[a][f"{v}_year_ext"]
            if b is None:
                draw(axes[r, c], x.mean("year"), f"{lab} {EXT[v]}", cmap=CMAP[v], lim=clim_lim(x.mean("year"), v), label=UNITS[v])
            else:
                y = ONE_D[b][f"{v}_year_ext"]
                draw(axes[r, c], diff(x.mean("year"), y.mean("year"), v), f"{lab} {EXT[v]}", cmap=DCMAP[v], sym=True,
                     sig=welch_sig(x, y), label=dlabel(v))
    plt.tight_layout(); plt.savefig(OUT / "extremes.png", dpi=110); plt.show()

## 10. Dry days and wet-day intensity

The WDF step acts on the dry-day frequency, the pr QDM on wet-day intensities. Dry-day differences in percentage points,
intensity in %. (At 0.7 deg a GCM has more drizzle days than ERA5 at 0.25 deg regridded: part of the bias is resolution.)

In [ ]:
ROWS_D = [r for r in (("gcm_historical", None, "historical"), ("gcm_historical", "era5", "historical - ERA5"),
                      ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
                      ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original"))
          if has(ONE_D.get(r[0]), "pr", "year_dry") and (r[1] is None or has(ONE_D.get(r[1]), "pr", "year_dry"))]
if ROWS_D:
    fig, axes = map_axes(len(ROWS_D), 2)
    for r, (a, b, lab) in enumerate(ROWS_D):
        dry_a, int_a = ONE_D[a]["pr_year_dry"], ONE_D[a]["pr_year_sdii"]
        if b is None:
            draw(axes[r, 0], dry_a.mean("year"), f"{lab}: dry-day fraction", cmap="YlOrBr", lim=(0, 1))
            draw(axes[r, 1], int_a.mean("year"), f"{lab}: wet-day intensity", cmap="YlGnBu", lim=clim_lim(int_a.mean("year"), "pr"), label="mm/day")
        else:
            dry_b, int_b = ONE_D[b]["pr_year_dry"], ONE_D[b]["pr_year_sdii"]
            draw(axes[r, 0], 100 * (dry_a.mean("year") - dry_b.mean("year")), f"{lab}: dry-day fraction", cmap="BrBG_r",
                 sym=True, sig=welch_sig(dry_a, dry_b), label="pp")
            draw(axes[r, 1], diff(int_a.mean("year"), int_b.mean("year"), "pr"), f"{lab}: wet-day intensity", cmap="BrBG",
                 sym=True, sig=welch_sig(int_a, int_b), label="%")
    plt.tight_layout(); plt.savefig(OUT / "dry_days_intensity.png", dpi=110); plt.show()

## 11. Pooled distributions (area-weighted random pixels of the domain, all days)

Quantiles of the daily values pooled over `N_SAMPLE` pixels of the domain of each dataset (ERA5 at 0.25 deg: finer pixels,
so naturally more extreme). Right: difference to the historical run, quantile by quantile.

In [ ]:
cols = {}
for n, ds in S.items():
    d = {v: ds[f"{v}_q"].values for v in VARS_PLOT if f"{v}_q" in ds}
    if "pr_q_wet" in ds:
        d["pr wet days"] = ds["pr_q_wet"].values
    cols[LABEL[n]] = pd.DataFrame(d, index=QUANTILES)
qtab = pd.concat(cols, axis=1); qtab.index.name = "quantile"
qtab.to_csv(OUT / "pooled_quantiles.csv")
display(qtab.round(2))

keys = [k for k in ("pr wet days", "tasmax", "tasmin", "dtr", "rsds") if any(k in df for df in cols.values())]
if "gcm_historical" in S and keys:
    fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.8), squeeze=False)
    for ax, k in zip(axes[0], keys):
        key = "pr_q_wet" if k == "pr wet days" else f"{k}_q"
        if key not in S["gcm_historical"]:
            ax.set_visible(False); continue
        ref = S["gcm_historical"][key].values
        for name, ds in S.items():
            if name != "gcm_historical" and key in ds:
                ax.plot(QUANTILES, ds[key].values - ref, "o-", ms=3, color=COLORS[name], label=LABEL[name])
        ax.set_xscale("logit"); ax.axhline(0, color="grey", lw=0.5)
        ax.set_title(f"{k}: dataset - historical"); ax.set_xlabel("quantile")
    axes[0, 0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(OUT / "pooled_quantile_differences.png", dpi=110); plt.show()

## 12. Interannual variability

Standard deviation of the yearly means: ratio historical / ERA5 (same period, same grid) and hosing / original (1 = no change).
A change of variability is not corrected by a quantile mapping calibrated on another run: good to know.

In [ ]:
ROWS_V = [r for r in (("gcm_historical", "era5", "historical / ERA5"), ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing / original"),
                      ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing / original")) if r[0] in ONE_D and r[1] in ONE_D]
vv = [v for v in ("pr", "tasmax", "tasmin", "rsds") if any(has(ONE_D[a], v) and has(ONE_D[b], v) for a, b, _ in ROWS_V)]
if ROWS_V and vv:
    fig, axes = map_axes(len(ROWS_V), len(vv))
    for r, (a, b, lab) in enumerate(ROWS_V):
        for c, v in enumerate(vv):
            if not (has(ONE_D[a], v) and has(ONE_D[b], v)):
                axes[r, c].set_visible(False); continue
            ratio = ONE_D[a][f"{v}_year_mean"].std("year") / ONE_D[b][f"{v}_year_mean"].std("year")
            draw(axes[r, c], ratio, f"{lab}: std of yearly {v}", cmap="PuOr_r", lim=(0.5, 1.5))
    plt.tight_layout(); plt.savefig(OUT / "interannual_variability_ratio.png", dpi=110); plt.show()

## Things to note (to fill in)

- Integrity / alerts: ...
- Raw bias of EC-Earth3 historical against ERA5 (where, which season, which variable): ...
- Stationarity of the 2025 / 2050 runs: ...
- Where the hosing signal is strong / significant (incl. extremes, dry days): ...
- Consequences for the downscaling / bias adjustment: ...

Figures and tables: `001_exploration/outputs/01_exploration/` (not in git). Statistics cache: `CACHE` (EOS).